In [2]:
import math
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pickle 
import networkx as nx
from collections import Counter
import statistics
import pdb
import openpyxl
import itertools
import os

## General Comments-
 For network files (represented by the graphml format or any of those files that can be read by networkx module)   that are upwards of 50 MB- processing times can go from 12-24 hours on a AMD EPYC CPU. 

# Module 1- Calculate Network for Branching Angle analysis

 Run this module first to create a network necessary for calculating branching angles
 Reference- Spangenberg P, Hagemann N, Squire A, Förster N, Krauß SD, Qi Y, Yusuf AM, Wang J, Grüneboom A, Kowitz L,    Korste S. Rapid and fully automated blood vasculature analysis in 3D light-sheet image volumes of different organs. Cell Reports Methods. 2023 Mar 27;3(3).

In [ ]:
filename= ''
skeleton_image= tifffile.imread(filename)
skeleton_image= np.where(skeleton_image == 255, 1,0 )
LIST_STEP_DIRECTIONS3D = list(itertools.product((-1, 0, 1), repeat=3))

In [ ]:
def _get_increments(config_number, dimensions):
    """
    Return position of non zero voxels/pixels in the
    binary string of config number
    Parameters
    ----------
    config_number : int64
        integer less than 2 ** 26

    dimensions: int
        number of dimensions, can only be 2 or 3

    Returns
    -------
    list
        a list of incremental direction of a non zero voxel/pixel

    Notes
    ------
    As in the beginning of the program, there are incremental directions
    around a voxel at origin (0, 0, 0) which are returned by this function.
    config_number is a decimal number representation of 26 binary numbers
    around a voxel at the origin in a second ordered neighborhood
    """
    config_number = np.int64(config_number)
    if dimensions == 3:
        # convert decimal number to a binary string
        list_step_directions = LIST_STEP_DIRECTIONS3D
    elif dimensions == 2:
        list_step_directions = LIST_STEP_DIRECTIONS2D
    neighbor_values = [(config_number >> digit) & 0x01 for digit in range(3 ** dimensions - 1)]
    return [neighbor_value * increment for neighbor_value, increment in zip(neighbor_values, list_step_directions)]

In [ ]:
def _set_adjacency_list(arr):
    """
    Return position of non zero voxels/pixels in the
    binary string of config number
    Parameters
    ----------
    arr : numpy array
        binary numpy array can only be 2D Or 3D

    Returns
    -------
    dict_of_indices_and_adjacent_coordinates: Dictionary
        key is the nonzero coordinate in input "arr" and value
        is all the position of nonzero coordinates around it
        in it's second order neighborhood

    """
    dimensions = arr.ndim
    assert dimensions in [2, 3], "array dimensions must be 2 or 3, they are {}".format(dimensions)
    if dimensions == 3:
        # flipped 3D template in advance
        template = np.array([[[33554432, 16777216, 8388608], [4194304, 2097152, 1048576], [524288, 262144, 131072]],
                            [[65536, 32768, 16384], [8192, 0, 4096], [2048, 1024, 512]],
                            [[256, 128, 64], [32, 16, 8], [4, 2, 1]]], dtype=np.uint64)
    else:
        # 2 dimensions
        template = np.array([[128, 64, 32], [16, 0, 8], [4, 2, 1]], dtype=np.uint64)
    # convert the binary array to a configuration number array of same size
    # by convolving with template
    arr = np.ascontiguousarray(arr, dtype=np.uint64)
    result = convolve(arr, template, mode='constant', cval=0)
    # set the values in convolution result to zero which were zero in 'arr'
    result[arr == 0] = 0
    dict_of_indices_and_adjacent_coordinates = {}
    # list of nonzero tuples
    non_zeros = list(set(map(tuple, np.transpose(np.nonzero(arr)))))
    if np.sum(arr) == 1:
        # if there is just one nonzero element there are no adjacent coordinates
        dict_of_indices_and_adjacent_coordinates[non_zeros[0]] = []
    else:
        for item in non_zeros:
            adjacent_coordinate_list = [tuple(np.array(item) + np.array(increments))
                                        for increments in _get_increments(result[item], dimensions) if increments != ()]
            dict_of_indices_and_adjacent_coordinates[item] = adjacent_coordinate_list
    return dict_of_indices_and_adjacent_coordinates


In [ ]:
dict_of_indices_and_adjacent_coordinates= _set_adjacency_list(skeleton_image)
G = nx.from_dict_of_lists(dict_of_indices_and_adjacent_coordinates)

In [ ]:
## Save network data acquired from vessel segmentation
nx.write_graphml(G, filename+'.graphml')

# Module 2- Calculate list of branch vectors and their pairs

 1. It is important to have the filenames be the same for the Branch_information.csv file obtained from the Fiji Analyze_Skeleton plugin and for the network generated in module 1. 
 2. The script assumes that the graphml file for the network and Branch_information.csv files are in the same folder as the script itself.
 3. Need to create a folder titled as branching_angles where the script will save all the branching angle data

In [2]:
filenames= ['VK-AA767_shaft_0-619_skeleton']
    

In [3]:
def angle_scripter(filename):
    branch_hist_path= os.path.abspath(filename+'.csv')
    graph_hist_path= os.path.abspath(filename + '.graphml')
    
    # Shaft_df is obtained from the 'Branch information.csv' file from the Analyze_Skeleton plugin from Fiji.
    shaft_df= pd.read_csv(branch_hist_path)
    shaft_nx= nx.read_graphml(graph_hist_path)
    
    branching_vectors= vector_enumerator(shaft_nx, shaft_df)
    pruned_pairs= branch_pruner(branching_vectors)
    branching_angles= calculate_branching_angles(pruned_pairs)
    
    angle_df= pd.DataFrame({'branching_angles': branching_angles})
    brangle_hist_path= os.path.abspath('branching_angles')
    angle_df.to_excel(brangle_hist_path+ '/'+filename+'_brangles'+'.xlsx')

In [11]:
for filename in filenames:
    angle_scripter(filename)

## Module 3- Nested Functions for Implementation

 Important to execute this module first before implementing module 2

In [4]:
# Converts node entries in adjacency lists from string to numeric format.
def string_to_coordinate(string):
    string= string.replace('(','')
    string= string.replace(')','')
    string= string.replace(',','')
    string= string.split(' ')
    X= int(string[0])
    Y= int(string[1])
    Z= int(string[2])
    return X, Y, Z

In [5]:
def DFS_search(adj_dic, start_node, max_depth=5):
    """
    Performs a depth-first search on a graph (given by an adjacency dictionary)
    starting from `start_node`. The search stops when a depth of `max_depth` is reached
    or when a node has no unvisited children (i.e. a leaf node).

    Parameters:
        adj_dic (dict): A dictionary where keys are node identifiers and values are iterables
                        of neighbor (child) nodes.
        start_node: The node from which to start the DFS.
        max_depth (int): Maximum number of nodes (levels) to traverse. A value of 5 means that
                         the DFS will traverse 5 nodes deep.
    
    Returns:
        endpoints (list): A list of nodes that were reached at the maximum depth or were leaves.
    Reference:
    Tarjan R. Depth-first search and linear graph algorithms. SIAM journal on computing. 1972 Jun;1(2):146-60.
    """
    
    
    endpoints = []    # This will store nodes at depth==max_depth or leaf nodes.
    visited = set()   # Use a set for efficiency.
    
    def dfs(node, depth):
        visited.add(node)
        # Get the unvisited children of the current node.
        children = [child for child in adj_dic.get(node, []) if child not in visited]
        
        # If we've reached max depth or there are no children (leaf node),
        # record the current node as an endpoint.
        if depth == max_depth or not children:
            endpoints.append(node)
            return
        
        # Otherwise, traverse each child.
        for child in children:
            dfs(child, depth + 1)
    
    dfs(start_node, 1)  # start at depth 1
    return endpoints

In [6]:
# Uses a depth-first search strategy to generate vectors that are 25 micrometers in length. Each vector points in the direction of a particular branch.
# 25 micrometers is chosen because it is the average diameter of a vessel. 
def endpoint_generator(G, node_entry):
    adj_dic= dict(G.adj)
    endpoints= DFS_search(adj_dic, node_entry, max_depth= 5)
    return endpoints

In [7]:
# Generates a list of tuples of size 2 consisting of vector pairs 
# where a branching angle can be calculated from each vector
# pair. 
def vector_enumerator(G, shaft_df):
    # The branch_information.csv file is used to filter out branches greater than
    # 25 micrometers in length
    filtered_shaft_df= shaft_df[shaft_df['Branch length'] >= 25]
    startingpoints_bin= []
    endingpoints_bin= []
    for i in filtered_shaft_df.iloc():
        startingpoints_bin.append((i['V1 x'], i['V1 y'], i['V1 z']))
        endingpoints_bin.append((i['V2 x'], i['V2 y'], i['V2 z']))
    # Here, all branch endpoints in the 'V1' and 'V2' column of the 
    # branch_information.csv file are concatenated into a single master list.
    
    list_all_points = []
    list_all_points.extend(startingpoints_bin)
    list_all_points.extend(endingpoints_bin)
    all_points_normalized= []
    
    # Here, the lateral resolution (x and y-axis) had a resolution of 2.6 micrometers 
    # and the axial resolution (z-axis) is 5 micrometers. All the pixels from the 
    # fiji branch_information.csv file needs to be divided by the resolution in order 
    # to obtain the individual voxel coordinates. 
    for k in list_all_points:
        x_normalized= round(k[0]/2.6) #X-resolution- lateral resolution
        y_normalized= round(k[1]/2.6) #Y-resolution- lateral resolution
        z_normalized= int(k[2]/5.0) #Z-resolution- axial resolution
        # Need to invert the order of the coordinates because in networkx the order is different.
        all_points_normalized.append((z_normalized, y_normalized, x_normalized))
    #Generate a list of unique points and eliminate any redundancies. 
    all_points_normalized= list(set(all_points_normalized))
    
    # Branching vector pairs are generated. One branching vector is used to cross-reference 
    # its complementary partner vector via the endpoint generator. 
    
    branching_vectors= []
    for j in all_points_normalized:
        node_entry= str(j)
        endpoints= endpoint_generator(G, node_entry)
        Z_origin, Y_origin, X_origin= string_to_coordinate(node_entry)
        for jj in endpoints:
            Z_target, Y_target, X_target= string_to_coordinate(jj)
            pair= [(Z_origin, Y_origin, X_origin), (Z_target, Y_target, X_target)]
            branching_vectors.append(pair)
     
    
    return branching_vectors

In [8]:
# Uses the resolution to convert from pixel co-ordinates to the micrometer domain in 3-dimensional space
def pixel_to_dist(p):
    z_d= 5*p[0]
    y_d= 2.6*p[1]
    x_d= 2.6*p[2]
    return (z_d, y_d, x_d)

In [9]:
# Identifies branches which are less than 25 micrometers apart and combines them into one. This prevents aberrant angle calculation as sometimes these branches appear as
# a result of noise from the skeletonization process.

def branch_pruner(branching_vectors):
    # Nexus Point refers to the common starting point of all vectors within a span of 25 micrometers.
    nexus_points= []
    for x in branching_vectors:
        nexus_points.append(x[0])
    nexus_points= list(set(nexus_points))
    
    # This will be the final bin where all vectors after pruning are stored. Vectors within a span of 25 
    # micrometers are pruned to a single vector. 
    pruned_pairs= []
    for n in nexus_points:
        # Finds a list of vectors with a common starting point n. 
        pairs= [x[1] for x in branching_vectors if x[0] == n]
        # The pair coordinates contain all possible vectors within 25 micrometers. 
        pair_coordinates= []
        target_point= pixel_to_dist(n)
        for p in pairs:
            d= pixel_to_dist(p)
            pair_coordinates.append([d, 'a'])
            
        while (len(pair_coordinates) > 0):
            # If only a single vector exists within a span of 25 micrometers, this is added to
            # the pruned_pairs bin and the cycle progesses to the next datapoint.
            if (len(pair_coordinates) == 1):
                pruned_pairs.append([target_point, pair_coordinates[0][0]])
                pair_coordinates.remove(pair_coordinates[0])
                break
            # To enumerate and mark branch-points which fall within a certain radius (25 micrometers)
            limit= 1
            pair_coordinates[0][1]= 'k'
            for j in range(limit, len(pair_coordinates)):
                d1z= pair_coordinates[0][0][0]
                d2z= pair_coordinates[j][0][0]
                d1y= pair_coordinates[0][0][1]
                d2y= pair_coordinates[j][0][1]
                d1x= pair_coordinates[0][0][2]
                d2x= pair_coordinates[j][0][2]
                dist= np.sqrt((d1x-d2x)**2+(d1y-d2y)**2+(d1z-d2z)**2)
                if (dist < 25):
                    pair_coordinates[j][1]= 'k'
            
            
            # List of points below 25 micrometers that need to be merged together- either median or mean.
            # Here the letter 'k' marks vectors to be pruned. The letter 'a' will mark a vector that does
            # not need to be pruned. If all vectors are below 25 micrometers, then they will all be 
            # converted to the letter 'k' and be pruned. 
            merge_points= [h for h in pair_coordinates if h[1] == 'k']
            pair_coordinates= [g for g in pair_coordinates if g[1] == 'a']
            # existing pair coordinates
            if len(merge_points) == 1:
                pruned_pairs.append([target_point, merge_points[0][0]])
            else:
                x_values= [m[0][2] for m in merge_points]
                y_values= [m[0][1] for m in merge_points]
                z_values= [m[0][0] for m in merge_points]
                x_median= statistics.median(x_values)
                y_median= statistics.median(y_values)
                z_median= statistics.median(z_values)
                # The pruned vectors are replaced by a single vector that is the median of all pruned vectors..
                pruned_pairs.append([target_point, (z_median, y_median, x_median)])   
    
    return pruned_pairs
    
    

In [10]:
# Input for function has to be pair values and not single entries as it is hard to link disjoint
# values of a pair together- need to undergo more iterations that will take extra time. 
def calculate_branching_angles(pruned_pairs): 
    
    # store branchpoints in a suitable container
    branch_points= {}
    points= []
    for k in pruned_pairs:
        points.append(k[0])

    # Generates branching vectors from pairs of endpoints.
    
    # Here, all starting endpoints for all vectors are stored in a bin.
    unique_branches= list(set(points))
    for i in unique_branches:
        branch_points.update({i:[]})
    # Branchpoints are stored as a dictionary of 
    # key-value pairs with the branchpoint (key) pointing to a list of endpoints (value).
    # This is later used to create the branching vectors.
    for f in pruned_pairs:
        branch_points[f[0]].append(f[1])     

    # Branchpoints with less than 2 branches are deleted.
    for kv in unique_branches:
        if (len(branch_points[kv]) == 1):
            branch_points.pop(kv)           
    
   
    branching_angles=[]
    
    for bp, endpoints in branch_points.items():
        # Compute branch vectors (using endpoint - branch_point)
        vectors = [(ep[0]-bp[0], ep[1]-bp[1], ep[2]-bp[2]) for ep in endpoints]
        
        # Iterate over all unique pairs of vectors.
        for vec1, vec2 in itertools.combinations(vectors, 2):
            vec1 = np.array(vec1)
            vec2 = np.array(vec2)
            norm1 = np.linalg.norm(vec1)
            norm2 = np.linalg.norm(vec2)
            if norm1 == 0 or norm2 == 0:
                continue  # Skip any zero-length vector to avoid division by zero.
            costheta = np.dot(vec1, vec2) / (norm1 * norm2)
            costheta = np.clip(costheta, -1.0, 1.0)  # Avoid numerical issues.
            angle_degrees = np.degrees(np.arccos(costheta))
            branching_angles.append(angle_degrees)
    
    return branching_angles